In [2]:
import polars as pl

from datetime import date

from ohlc_dss_model.data.data_loader import load_parquet
from ohlc_dss_model.data.integrity import remove_incomplete_days
from ohlc_dss_model.data.tagging import intraday_session_tagging, session_tagging

from ohlc_dss_model.features.session_aggregation import aggregate_sessions

from ohlc_dss_model.utils.dt_utils import convert_to_timezone


In [3]:
df = load_parquet()
df = convert_to_timezone(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)

print(df.head(5))

shape: (5, 8)
┌──────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┬─────────────────┐
│ DateTime         ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume ┆ Session    ┆ Intraday_Sessio │
│ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        ┆ n               │
│ datetime[ns, Ame ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    ┆ date       ┆ ---             │
│ rica/New_York]   ┆         ┆         ┆         ┆         ┆        ┆            ┆ str             │
╞══════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╪═════════════════╡
│ 2016-01-03       ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   ┆ 2016-01-04 ┆ Asia            │
│ 18:00:00 EST     ┆         ┆         ┆         ┆         ┆        ┆            ┆                 │
│ 2016-01-03       ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    ┆ 2016-01-04 ┆ Asia            │
│ 18:30:00 EST     ┆         ┆         ┆         ┆         ┆        ┆        

We need to group our data by its session such that each row represents one trading day

We will not feed the model the raw intraday data, instead we will summarize those into features.

In [4]:
session_df = (
    df.group_by(["Session", "Intraday_Session"])
    .agg([
        pl.col("Open").first().alias("O"),
        pl.col("High").max().alias("H"),
        pl.col("Low").min().alias("L"),
        pl.col("Close").last().alias("C"),
    ])
    
    .pivot(
        index="Session",
        on="Intraday_Session",
        values=["O", "H", "L", "C"],
    )
    .sort("Session")
)
print(session_df.head(5))
print(session_df.shape)
print(session_df.columns)


shape: (5, 13)
┌────────────┬─────────┬────────────┬──────────┬───┬──────────┬─────────┬────────────┬──────────┐
│ Session    ┆ O_Asia  ┆ O_New York ┆ O_London ┆ … ┆ L_London ┆ C_Asia  ┆ C_New York ┆ C_London │
│ ---        ┆ ---     ┆ ---        ┆ ---      ┆   ┆ ---      ┆ ---     ┆ ---        ┆ ---      │
│ date       ┆ f64     ┆ f64        ┆ f64      ┆   ┆ f64      ┆ f64     ┆ f64        ┆ f64      │
╞════════════╪═════════╪════════════╪══════════╪═══╪══════════╪═════════╪════════════╪══════════╡
│ 2016-01-04 ┆ 4592.5  ┆ 4498.75    ┆ 4529.75  ┆ … ┆ 4483.75  ┆ 4529.5  ┆ 4506.25    ┆ 4498.25  │
│ 2016-01-05 ┆ 4505.5  ┆ 4494.75    ┆ 4509.0   ┆ … ┆ 4466.0   ┆ 4509.25 ┆ 4481.5     ┆ 4494.75  │
│ 2016-01-06 ┆ 4482.0  ┆ 4396.25    ┆ 4445.0   ┆ … ┆ 4386.0   ┆ 4444.5  ┆ 4448.5     ┆ 4396.25  │
│ 2016-01-07 ┆ 4450.25 ┆ 4316.0     ┆ 4365.0   ┆ … ┆ 4297.25  ┆ 4365.0  ┆ 4293.25    ┆ 4316.75  │
│ 2016-01-08 ┆ 4298.0  ┆ 4337.0     ┆ 4339.75  ┆ … ┆ 4309.0   ┆ 4340.0  ┆ 4264.5     ┆ 4335.5   │
└────

In [5]:
features = aggregate_sessions(df)
print(features.head(5))

shape: (5, 13)
┌────────────┬─────────┬──────────┬────────────┬───┬────────────┬─────────┬──────────┬────────────┐
│ Session    ┆ O_Asia  ┆ O_London ┆ O_New York ┆ … ┆ L_New York ┆ C_Asia  ┆ C_London ┆ C_New York │
│ ---        ┆ ---     ┆ ---      ┆ ---        ┆   ┆ ---        ┆ ---     ┆ ---      ┆ ---        │
│ date       ┆ f64     ┆ f64      ┆ f64        ┆   ┆ f64        ┆ f64     ┆ f64      ┆ f64        │
╞════════════╪═════════╪══════════╪════════════╪═══╪════════════╪═════════╪══════════╪════════════╡
│ 2016-01-04 ┆ 4592.5  ┆ 4529.75  ┆ 4498.75    ┆ … ┆ 4429.75    ┆ 4529.5  ┆ 4498.25  ┆ 4506.25    │
│ 2016-01-05 ┆ 4505.5  ┆ 4509.0   ┆ 4494.75    ┆ … ┆ 4457.25    ┆ 4509.25 ┆ 4494.75  ┆ 4481.5     │
│ 2016-01-06 ┆ 4482.0  ┆ 4445.0   ┆ 4396.25    ┆ … ┆ 4391.0     ┆ 4444.5  ┆ 4396.25  ┆ 4448.5     │
│ 2016-01-07 ┆ 4450.25 ┆ 4365.0   ┆ 4316.0     ┆ … ┆ 4283.0     ┆ 4365.0  ┆ 4316.75  ┆ 4293.25    │
│ 2016-01-08 ┆ 4298.0  ┆ 4339.75  ┆ 4337.0     ┆ … ┆ 4255.75    ┆ 4340.0  ┆ 4335.5   